# MuonClip: saved initialization vs saved final angular spectrum

This is the canonical initial-to-final angular WeightWatcher analysis for the one-head NanoGPT baseline.

It loads the **actual** `checkpoint_initial.pt` (step 0) and `checkpoint_final.pt`, extracts all six matrices through WeightWatcher, computes the gauge-invariant tilt/twist angular spectra, and compares them with matched random-angular nulls.

## Power-law tail contract

The angular power-law test deliberately follows the same `powerlaw` package logic used by WeightWatcher:

```python
fit = powerlaw.Fit(all_positive_projective_angular_values,
                   discrete=False, verbose=False)
```

We pass **no `xmin` and no `xmax`**. The package performs its MLE/KS search for the start of the tail, and the fit contains every observed value from the selected `xmin` through the **largest observed angular value**. `ANGULAR_MIN_TAIL` is checked only after the package has selected `xmin`; it never chooses the fit window.

For each layer and angular sector the notebook makes package-native PDF, CDF, and CCDF plots plus a dedicated **far-tail zoom** comparing the trained tail with the randomized-angular 95% envelope. The zoom changes only the plot limits; it does not truncate the data given to `powerlaw.Fit`.

Environment variables must be exported before Jupyter starts:

```bash
cd /path/to/rg_optimizers
export RG_OPTIMIZERS_ROOT="$PWD"
export RUNROOT=/tmp/<same-run-root-used-by-training>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"

# More null realizations for a serious comparison:
export ANGULAR_N_NULL=500

jupyter lab baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb
```


In [ ]:
from pathlib import Path
import os
import sys

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch Jupyter from the rg_optimizers repository"
    )

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)


In [ ]:
from IPython.display import display
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_weightwatcher_pipeline import run_analysis
from rg_nanogpt_one_head.angular_powerlaw_tail import run_powerlaw_tail_analysis

CONFIG = AnalysisConfig.from_env()
print(CONFIG)


In [ ]:
# Standard initial-vs-final angular/radial analysis.
RESULTS, RESOLVED_RUN, ANALYSIS_MANIFEST = run_analysis(CONFIG)
print("RUN_DIR =", RESOLVED_RUN.run_dir, "via", RESOLVED_RUN.run_dir_source)
print("INITIAL =", RESOLVED_RUN.initial_path)
print("FINAL =", RESOLVED_RUN.final_path)
print("OUTPUT =", RESOLVED_RUN.output_dir)
display(RESULTS)


In [ ]:
# Canonical far-tail test.
#
# This reruns the angular null generation so that the tail fit and far-tail
# plots have an explicit, auditable contract:
#   * all positive x are supplied to powerlaw.Fit
#   * xmin is not supplied
#   * xmax is not supplied
#   * the package selects xmin by its MLE/KS procedure
#   * every largest observed x >= xmin remains in the fit
TAIL_RESULTS, _ = run_powerlaw_tail_analysis(CONFIG)
display(TAIL_RESULTS)

print("Tail summary:")
print(RESOLVED_RUN.output_dir / "angular_powerlaw_far_tail_summary.csv")
print("Key plots per layer/sector:")
print("  *_powerlaw_pdf_loglog.png")
print("  *_powerlaw_pdf_linear.png")
print("  *_powerlaw_cdf.png")
print("  *_powerlaw_ccdf_loglog.png")
print("  *_far_tail_zoom_ccdf.png")


## Interpretation

The scientific test is **not** “does a log-log plot look straight?” A random angular matrix already has nontrivial edge/tail geometry.

Evidence for nonrandom angular RG/scale-free behavior requires the trained far tail to differ from the matched Haar/Stiefel null. In particular inspect:

- `actual_alpha` versus the null alpha interval;
- the package-selected `actual_xmin`;
- `actual_tail_n`;
- `actual_tail_decades = log10(xmax_observed/xmin)`;
- whether the trained CCDF remains extended beyond the random-angular 95% envelope.

A fitted exponent alone is not sufficient evidence.
